# Docker para MLOps — Bella Tavola 🐳
## Parte 3: Integrando Docker ao pipeline de CI

## Como usar este caderno?

Este é o caderno final da série Docker. Em e05-p01 e e05-p02, você buildou, rodou e refinou a imagem da Bella Tavola localmente. Agora essa imagem entra no pipeline de CI.

O objetivo é direto: a cada merge no `main`, se os testes passarem, a imagem é publicada automaticamente no Docker Hub. O pipeline que você construiu na Semana 3 ganha um quarto job — e a promessa do final de e04-p03 finalmente se cumpre.

Todo o trabalho acontece no terminal, no editor e no navegador (GitHub e Docker Hub). As células de código são referências para estudar e adaptar.

Os gabaritos estão nas células colapsadas logo após cada exercício — tente resolver antes de abrir.

## O que é essencial, recomendado e desafio?

**Essencial:** o mínimo que você precisa conseguir fazer para completar a etapa.

**Recomendado:** amplia a qualidade da solução.

**Desafio 🎯:** aprofunda a solução do ponto de vista de engenharia.

## Pré-requisitos

Este é o caderno com mais dependências externas da série. Antes de começar, confirme **todos** os itens:

In [ ]:
# Execute no terminal, na raiz do projeto Bella Tavola:

# 1. O Dockerfile com multi-stage e usuário não-root existe?
# cat Dockerfile | grep -E 'AS builder|USER appuser'
# Saída esperada: duas linhas encontradas

# 2. O .dockerignore existe e o .env não entra na imagem?
# docker build -t bella-tavola:v3 . && \
# docker run --rm bella-tavola:v3 find /app -name '.env'
# Saída esperada: nenhuma saída (arquivo não existe na imagem)

# 3. O docker-compose.yml sobe API + PostgreSQL + Nginx?
# docker compose up -d && sleep 5 && curl http://localhost/
# Saída esperada: resposta JSON da Bella Tavola
# docker compose down

# 4. O pipeline de CI da Semana 3 está verde?
# Abra: https://github.com/seu-usuario/bella-tavola/actions
# Confirme: os três jobs (qualidade, integracao, relatorio) estão com ✅

# 5. Você tem conta no Docker Hub e o repositório bella-tavola criado?
# Abra: https://hub.docker.com/r/seu-usuario/bella-tavola
# Se não existir, crie antes de continuar (instruções no Exercício 13.1)

# Se qualquer item falhar, resolva antes de continuar.
# O Bloco 13 depende de todos esses fundamentos funcionando.

---

# BLOCO 13 — Integrando Docker ao pipeline de CI

> **Objetivo:** Adicionar um job `docker` ao `ci.yml` existente que, a cada merge no `main`, faz build da imagem e push para o Docker Hub — completando o pipeline `qualidade → integracao → docker → relatorio`.

## Conceitos-chave do Bloco 13

### Docker Hub como registry de imagens

Na Semana 2, você usou o **Hugging Face Hub** como registry de modelos — um lugar central para armazenar, versionar e distribuir o artefato `.pkl`. O **Docker Hub** é o equivalente para imagens de contêiner.

| Conceito | Hugging Face Hub (Semana 2) | Docker Hub (esta semana) |
|---|---|---|
| O que armazena | Modelos (`.pkl`, `README.md`) | Imagens de contêiner |
| Convenção de nome | `usuario/modelo` | `usuario/repositorio:tag` |
| Como enviar | `api.upload_file(...)` | `docker push usuario/repo:tag` |
| Como receber | `hf_hub_download(...)` | `docker pull usuario/repo:tag` |
| Versionamento | Commits no Hub | Tags da imagem |

### Tags como estratégia de versionamento

No exercício 9.3, você viu que `:latest` é uma tag convencional — não é atualizada automaticamente. Em produção, a estratégia mais confiável é usar o **hash do commit Git** como tag:

```
usuario/bella-tavola:a3f2b1c9d8e7f6a5b4c3d2e1f0a9b8c7
                     ↑
                     github.sha — identifica exatamente qual commit gerou essa imagem
```

Com isso, dada qualquer imagem em produção, você sabe exatamente qual versão do código ela representa. Esse rastreamento é fundamental em MLOps — onde código, modelo e imagem precisam estar alinhados.

### O pipeline atual e onde o job `docker` se encaixa

O pipeline que você construiu na Semana 3 tem três jobs:

```
Push ou PR para main
        ↓
  ┌─────────────┐
  │  qualidade  │  → formatação + testes smoke      (roda em todo push e PR)
  └──────┬──────┘
         │ (só em push para main)
         ↓
  ┌─────────────┐
  │ integracao  │  → modelo do Hub + testes de integração
  └──────┬──────┘
         │
         ↓
  ┌─────────────┐
  │  relatorio  │  → resumo do pipeline
  └─────────────┘
```

O job `docker` entra entre `integracao` e `relatorio`:

```
Push ou PR para main
        ↓
  ┌─────────────┐
  │  qualidade  │  → formatação + testes smoke
  └──────┬──────┘
         │ (só em push para main)
         ↓
  ┌─────────────┐
  │ integracao  │  → modelo do Hub + testes de integração
  └──────┬──────┘
         │
         ↓
  ┌─────────────┐
  │   docker    │  → build da imagem + push para Docker Hub  ← novo
  └──────┬──────┘
         │
         ↓
  ┌─────────────┐
  │  relatorio  │  → resumo do pipeline
  └─────────────┘
```

A lógica é: se os testes de integração passaram, o código e o modelo estão íntegros — a imagem pode ser publicada com confiança.

### Actions do Docker para GitHub Actions

Você poderia fazer build e push com `docker build` e `docker push` diretamente no step `run:`. Mas o ecossistema Docker publicou actions oficiais no GitHub Marketplace que fazem isso de forma mais robusta — com cache de layers, autenticação segura e suporte a múltiplas plataformas:

| Action | O que faz |
|---|---|
| `docker/setup-buildx-action@v3` | Configura o BuildKit — o builder moderno do Docker, necessário para cache no CI |
| `docker/login-action@v3` | Autentica no Docker Hub (ou outro registry) de forma segura com secrets |
| `docker/build-push-action@v5` | Faz build e push da imagem com cache de layers integrado |

### Cache de build no CI

Sem cache, cada execução do pipeline refaz o build do zero — incluindo o `pip install` que instala scikit-learn e numpy. Isso pode levar 5 a 10 minutos por build.

O `docker/build-push-action` suporta cache via GitHub Actions Cache:

```yaml
- uses: docker/build-push-action@v5
  with:
    cache-from: type=gha          # lê cache do GitHub Actions
    cache-to: type=gha,mode=max   # salva cache no GitHub Actions
```

Na prática, isso significa que se o `requirements.txt` não mudou, o `pip install` usa o cache e o build fica segundos — não minutos.

### Autenticação com secrets

O mesmo padrão de secrets que você usou para o `HF_TOKEN` na Semana 3 se aplica aqui. Você vai criar dois novos secrets no repositório GitHub:

- `DOCKER_USERNAME` — seu nome de usuário no Docker Hub
- `DOCKER_PASSWORD` — um **token de acesso** gerado no Docker Hub (nunca use sua senha diretamente)

Tokens de acesso são revogáveis individualmente — se um token vazar, você o revoga sem precisar trocar a senha da conta.

## Exercício 13.1 — Criar conta, repositório e secrets

**Nível:** Essencial  
**Conceito:** Configurar os pré-requisitos externos antes de tocar no `ci.yml`.

### Parte A — Docker Hub

1. Acesse [hub.docker.com](https://hub.docker.com) e crie uma conta se ainda não tiver
2. Crie um repositório público chamado `bella-tavola`:
   - Clique em **Create repository**
   - Nome: `bella-tavola`
   - Visibilidade: **Public**
   - Clique em **Create**
3. Gere um token de acesso (nunca use sua senha no pipeline):
   - Vá em **Account Settings → Security → New Access Token**
   - Nome: `bella-tavola-ci`
   - Permissões: **Read & Write**
   - Copie o token — ele só aparece uma vez

### Parte B — Secrets no GitHub

1. Abra seu repositório no GitHub
2. Vá em **Settings → Secrets and variables → Actions**
3. Crie dois secrets:
   - `DOCKER_USERNAME`: seu nome de usuário no Docker Hub
   - `DOCKER_PASSWORD`: o token de acesso gerado no passo anterior

### Verificação

In [ ]:
# ✏️ Confirme antes de continuar:

# O repositório bella-tavola existe no Docker Hub?
# URL: https://hub.docker.com/r/seu-usuario/bella-tavola
# Sim no usuario anaclaragr

# Os dois secrets existem no GitHub?
# Settings → Secrets and variables → Actions
# Você deve ver DOCKER_USERNAME e DOCKER_PASSWORD na lista
# (os valores ficam ocultos — só os nomes aparecem)
# Sim

# Por que usar um token de acesso em vez da senha?
# Ele é mais seguro, pode ser revogado e permite dar permissões específicas

# O que acontece se o token vazar? Como você responderia?
# Existe vazamento de dados que podem ser prejudiciais dando informações
# para quem não deve
# Revogar o token

<details>
<summary>💡 Gabarito</summary>

```python
# Por que usar token em vez de senha:
# 1. Escopo limitado: o token só tem as permissões que você definiu (Read & Write)
#    Sua senha dá acesso total à conta — incluindo apagar repositórios
# 2. Revogabilidade: se o token vazar, você o revoga em Account Settings → Security
#    sem precisar trocar a senha (o que afetaria todos os seus outros serviços)
# 3. Rastreabilidade: você pode ter múltiplos tokens com nomes descritivos
#    e saber exatamente qual sistema está usando qual token
#
# É o mesmo raciocínio do HF_TOKEN da Semana 2 — tokens de acesso
# são a prática padrão da indústria para autenticação em pipelines.

# Se o token vazar (ex: commit acidental no repositório público):
# 1. IMEDIATAMENTE: revogue o token em Docker Hub → Account Settings → Security
#    O token comprometido para de funcionar instantaneamente.
# 2. Gere um novo token com o mesmo escopo
# 3. Atualize o secret DOCKER_PASSWORD no GitHub
# 4. Remova o commit com o token do histórico Git (git rebase ou filter-branch)
# 5. Verifique os logs de acesso no Docker Hub para atividade suspeita
```

</details>

## Exercício 13.2 — Build e push manual

**Nível:** Essencial  
**Conceito:** Entender cada etapa do processo manualmente antes de automatizar.

O mesmo princípio da Semana 3: antes de automatizar qualquer coisa, faça o processo manual para entender o que está sendo automatizado. Se o push manual falha, o pipeline vai falhar pelo mesmo motivo.

### Referência

```bash
# Execute na raiz do projeto Bella Tavola

# Passo 1: Autenticar no Docker Hub
docker login
# Saída esperada: solicitação de usuário e senha (use o token como senha)
# Login Succeeded

# Alternativa com variáveis (evita digitar interativamente):
# echo $DOCKER_PASSWORD | docker login -u $DOCKER_USERNAME --password-stdin

# Passo 2: Build com a tag no formato usuario/repositorio:tag
# Substitua 'seu-usuario' pelo seu usuário real do Docker Hub
docker build -t seu-usuario/bella-tavola:v1 .
# Saída esperada: build normal com os steps do Dockerfile

# Verificar que a tag foi criada:
docker images | grep bella-tavola
# Saída esperada: linha com seu-usuario/bella-tavola e tag v1

# Passo 3: Push para o Docker Hub
docker push seu-usuario/bella-tavola:v1
# Saída esperada: upload das layers
# Cada layer é enviada uma vez e reutilizada em pushes futuros
# v1: digest: sha256:... size: ...

# Passo 4: Verificar no Docker Hub
# Abra: https://hub.docker.com/r/seu-usuario/bella-tavola/tags
# A tag v1 deve aparecer com tamanho e data

# Passo 5: Simular 'outra máquina' — remover a imagem local e fazer pull
docker rmi seu-usuario/bella-tavola:v1
docker pull seu-usuario/bella-tavola:v1
# Saída esperada: download das layers do Docker Hub
# Pull complete

# Confirmar que funciona:
docker run -p 8000:8000 --rm --env-file .env seu-usuario/bella-tavola:v1
# Em outro terminal:
# curl http://localhost:8000/
```

### Sua tarefa

Execute a sequência completa: login, build com sua tag, push, verificação no Hub, remoção local, pull e execução.

In [ ]:
# ✏️ Anote:

# O push foi bem-sucedido?
# Foi bem sucedido

# Quanto tempo levou o primeiro push?
# O primeiro push demorou mais que o segundo push

# Agora faça uma pequena alteração no código (ex: mensagem da rota /)
# e repita o build + push com a tag :v2.
# O segundo push foi mais rápido? Por que?
# (dica: observe quais layers dizem 'Layer already exists')
# Foi mais rapido pq enviou apenas a layer nova, que ja existia.

# URL do seu repositório no Docker Hub:
# https://hub.docker.com/repository/docker/anaclaragr/bella-tavola/general

<details>
<summary>💡 Gabarito</summary>

```python
# O segundo push foi mais rápido porque o Docker Hub já tem as layers
# que não mudaram — especialmente a layer do pip install
# (python:3.11-slim + todos os pacotes instalados).
# O segundo push enviou apenas a layer nova (o COPY . . com o código alterado).
# 'Layer already exists' significa: essa layer já está no Hub,
# não precisa reenviar.
#
# Isso é o mesmo mecanismo de cache que você viu no build local:
# layers imutáveis são compartilhadas entre versões diferentes da imagem.
# Uma imagem de 1.1GB pode ser enviada em segundos na segunda vez
# porque 99% das layers já existem no registry.
```

</details>

## Exercício 13.3 — Adicionando o job `docker` ao ci.yml

**Nível:** Essencial  
**Conceito:** Estender o pipeline existente com um job que usa actions oficiais do Docker.

### Referência — o job `docker` completo

```yaml
# Trecho a adicionar no .github/workflows/ci.yml
# Posicione APÓS o job 'integracao' e ANTES do job 'relatorio'

  # ─────────────────────────────────────────────────────────────────
  # JOB docker
  # Roda apenas em push para o main — após os testes de integração.
  # Se os testes falharam, a imagem não é publicada.
  # Usa o hash do commit como tag para rastreabilidade.
  # ─────────────────────────────────────────────────────────────────
  docker:
    runs-on: ubuntu-latest
    needs: integracao
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'

    steps:
      - name: Baixar o código
        uses: actions/checkout@v4

      # Configura o BuildKit — o builder moderno do Docker
      # Necessário para o cache de layers funcionar no CI
      - name: Configurar Docker Buildx
        uses: docker/setup-buildx-action@v3

      # Autentica no Docker Hub usando os secrets do repositório
      # Nunca coloque usuário e senha diretamente no YAML
      - name: Autenticar no Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKER_USERNAME }}
          password: ${{ secrets.DOCKER_PASSWORD }}

      # Build e push da imagem
      # context: . → usa o Dockerfile na raiz do repositório
      # push: true → envia para o registry após o build
      # tags: duas tags — hash do commit (rastreabilidade) e latest (conveniência)
      # cache-from/to → evita rebuildar layers que não mudaram
      - name: Build e push da imagem
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: |
            ${{ secrets.DOCKER_USERNAME }}/bella-tavola:${{ github.sha }}
            ${{ secrets.DOCKER_USERNAME }}/bella-tavola:latest
          cache-from: type=gha
          cache-to: type=gha,mode=max
```

### O job `relatorio` precisa ser atualizado

O job `relatorio` atualmente tem `needs: integracao`. Agora que o job `docker` existe entre os dois, `relatorio` deve depender de `docker`:

```yaml
  relatorio:
    runs-on: ubuntu-latest
    needs: docker        # ← atualizado de 'integracao' para 'docker'
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v4
      - name: Resumo do pipeline
        run: |
          echo "================================================"
          echo "  Pipeline CI do Bella Tavola concluído ✅"
          echo "================================================"
          echo "  Branch:  ${{ github.ref_name }}"
          echo "  Commit:  ${{ github.sha }}"
          echo "  Autor:   ${{ github.actor }}"
          echo "  Imagem:  ${{ secrets.DOCKER_USERNAME }}/bella-tavola:${{ github.sha }}"
          echo "================================================"
```

### Sua tarefa

1. Abra `.github/workflows/ci.yml` no editor
2. Adicione o job `docker` conforme a referência acima
3. Atualize `needs: integracao` para `needs: docker` no job `relatorio`
4. Atualize a linha de resumo do `relatorio` para incluir a URL da imagem
5. Faça commit e push para o branch `main`
6. Acompanhe o pipeline na aba **Actions** do GitHub

In [ ]:
# ✏️ Acompanhe o pipeline e anote:

# Os quatro jobs apareceram na aba Actions?
# qualidade → integracao → docker → relatorio
# Sim!!

# O job 'docker' passou?
# Se falhou: em qual step? Qual foi a mensagem de erro?
# passou

# A imagem apareceu no Docker Hub com a tag do commit?
# URL: https://hub.docker.com/r/seu-usuario/bella-tavola/tags
# A tag deve ser os 40 caracteres do SHA do commit
#  sim

# Quanto tempo levou o job 'docker' no primeiro build (sem cache)?
# demou mais que o primeiro, um minuto

# Faça um segundo commit sem mudar o requirements.txt.
# Quanto tempo levou o segundo build (com cache)?
# demorou menos por conta do cache

name: CI - Bella Tavola

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  qualidade:
    runs-on: ubuntu-latest

    steps:
      - name: Baixar o código
        uses: actions/checkout@v4

      - name: Configurar Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Instalar dependências
        run: |
          python -m pip install --upgrade pip
          pip install -r aula_01_e02_fastapi/requirements.txt

      - name: Verificar formatação
        run: |
          cd aula_01_e02_fastapi
          python -m black --check .

      - name: Verificar imports não utilizados
        run: |
          cd aula_01_e02_fastapi
          python -m autoflake --check --remove-all-unused-imports -r .

      - name: Rodar testes smoke
        run: |
          cd aula_01_e02_fastapi
          python -m pytest tests/ -m "smoke or validacao" -v --tb=short

  integracao:
    runs-on: ubuntu-latest
    needs: qualidade
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'

    steps:
      - name: Baixar o código
        uses: actions/checkout@v4

      - name: Configurar Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Instalar dependências
        run: |
          python -m pip install --upgrade pip
          pip install -r aula_01_e02_fastapi/requirements.txt

      - name: Cache do modelo Hugging Face
        uses: actions/cache@v4
        with:
          path: ~/.cache/huggingface
          key: hf-model-v1-${{ hashFiles('aula_01_e02_fastapi/requirements.txt') }}

      - name: Verificar autenticação com Hugging Face
        run: python -c "import os; assert os.environ.get('HF_TOKEN'), 'HF_TOKEN não configurado'"
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}

      - name: Rodar testes de integração
        run: |
          cd aula_01_e02_fastapi
          python -m pytest tests/test_modelo.py -m integracao -v --tb=short
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_REPO_ID: anaclaragr05/mlops-fraud-v1

  docker:
    runs-on: ubuntu-latest
    needs: integracao
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'

    steps:
      - name: Baixar o código
        uses: actions/checkout@v4

      - name: Configurar Docker Buildx
        uses: docker/setup-buildx-action@v3

      - name: Autenticar no Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKER_USERNAME }}
          password: ${{ secrets.DOCKER_PASSWORD }}

      - name: Build e push da imagem
        uses: docker/build-push-action@v5
        with:
          context: ./aula_01_e02_fastapi
          push: true
          tags: |
            ${{ secrets.DOCKER_USERNAME }}/bella-tavola:${{ github.sha }}
            ${{ secrets.DOCKER_USERNAME }}/bella-tavola:latest
          cache-from: type=gha
          cache-to: type=gha,mode=max

  relatorio:
    runs-on: ubuntu-latest
    needs: docker
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'

    steps:
      - name: Resumo do pipeline
        run: |
          echo "================================================"
          echo "  Pipeline CI do Bella Tavola concluído ✅"
          echo "================================================"
          echo "  Branch:  ${{ github.ref_name }}"
          echo "  Commit:  ${{ github.sha }}"
          echo "  Autor:   ${{ github.actor }}"
          echo "  Imagem:  ${{ secrets.DOCKER_USERNAME }}/bella-tavola:${{ github.sha }}"
          echo "  Latest:  ${{ secrets.DOCKER_USERNAME }}/bella-tavola:latest"
          echo "================================================"

<details>
<summary>💡 Gabarito — ci.yml completo com job docker</summary>

```yaml
# .github/workflows/ci.yml — versão final com job docker
name: CI — Bella Tavola

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  # ─────────────────────────────────────────────────────
  # JOB 1: qualidade
  # Roda em todo push e PR. Sem dependências externas.
  # ─────────────────────────────────────────────────────
  qualidade:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Instalar dependências
        run: |
          pip install --upgrade pip
          pip install -r requirements.txt
      - name: Verificar formatação
        run: black --check .
      - name: Verificar imports não utilizados
        run: autoflake --check --remove-all-unused-imports -r .
      - name: Rodar testes smoke
        run: pytest tests/ -m smoke -v --tb=short

  # ─────────────────────────────────────────────────────
  # JOB 2: integracao
  # Só em push para main. Usa HF_TOKEN.
  # ─────────────────────────────────────────────────────
  integracao:
    runs-on: ubuntu-latest
    needs: qualidade
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Instalar dependências
        run: |
          pip install --upgrade pip
          pip install -r requirements.txt
      - name: Cache do modelo Hugging Face
        uses: actions/cache@v4
        with:
          path: ~/.cache/huggingface
          key: hf-model-v1-${{ hashFiles('requirements.txt') }}
      - name: Verificar autenticação com Hugging Face
        run: python -c "import os; assert os.environ.get('HF_TOKEN'), 'HF_TOKEN não configurado'"
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
      - name: Rodar testes de integração
        run: pytest tests/ -m integracao -v --tb=short
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}

  # ─────────────────────────────────────────────────────
  # JOB 3: docker
  # Só em push para main, após integracao passar.
  # Publica imagem no Docker Hub com tag do commit.
  # ─────────────────────────────────────────────────────
  docker:
    runs-on: ubuntu-latest
    needs: integracao
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    steps:
      - name: Baixar o código
        uses: actions/checkout@v4
      - name: Configurar Docker Buildx
        uses: docker/setup-buildx-action@v3
      - name: Autenticar no Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKER_USERNAME }}
          password: ${{ secrets.DOCKER_PASSWORD }}
      - name: Build e push da imagem
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: |
            ${{ secrets.DOCKER_USERNAME }}/bella-tavola:${{ github.sha }}
            ${{ secrets.DOCKER_USERNAME }}/bella-tavola:latest
          cache-from: type=gha
          cache-to: type=gha,mode=max

  # ─────────────────────────────────────────────────────
  # JOB 4: relatorio
  # Roda após docker — resumo com URL da imagem publicada.
  # ─────────────────────────────────────────────────────
  relatorio:
    runs-on: ubuntu-latest
    needs: docker
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v4
      - name: Resumo do pipeline
        run: |
          echo "================================================"
          echo "  Pipeline CI do Bella Tavola concluído ✅"
          echo "================================================"
          echo "  Branch:  ${{ github.ref_name }}"
          echo "  Commit:  ${{ github.sha }}"
          echo "  Autor:   ${{ github.actor }}"
          echo "  Imagem:  ${{ secrets.DOCKER_USERNAME }}/bella-tavola:${{ github.sha }}"
          echo "================================================"
          echo "  Próximo passo: deploy da imagem em produção"
          echo "================================================"
```

</details>

## Exercício 13.4 — Verificando a imagem publicada pelo pipeline

**Nível:** Recomendado  
**Conceito:** Fechar o ciclo — puxar a imagem publicada pelo CI e confirmar que funciona como a versão local.

### Referência

```bash
# Execute no terminal — em QUALQUER pasta, não precisa ser o projeto
# (simulando uma 'outra máquina' que não tem o código)

# Obter o SHA do último commit publicado
# (copie da aba Actions do GitHub ou do Docker Hub)
SHA=a3f2b1c9d8e7f6a5b4c3d2e1f0a9b8c7d6e5f4a3   # substitua pelo SHA real

# Baixar a imagem publicada pelo pipeline
docker pull seu-usuario/bella-tavola:$SHA
# Saída esperada: download das layers
# Digest: sha256:...
# Status: Downloaded newer image

# Rodar a imagem publicada
docker run -p 8000:8000 --rm \
  --env-file /caminho/para/o/seu/.env \
  seu-usuario/bella-tavola:$SHA

# Em outro terminal — testar que a API funciona:
curl http://localhost:8000/
curl http://localhost:8000/pratos
curl -X POST http://localhost:8000/ml/predict \
  -H "Content-Type: application/json" \
  -d '{"valor_transacao": 200.0, "hora_transacao": 10, \
       "distancia_ultima_compra": 3.0, "tentativas_senha": 1, \
       "pais_diferente": 0}'
```

### Sua tarefa

1. Obtenha o SHA do commit mais recente na aba Actions do GitHub
2. Faça pull da imagem publicada pelo pipeline
3. Rode e confirme que as rotas respondem corretamente
4. Responda as perguntas abaixo

In [ ]:
# ✏️ Anote:

# SHA do commit testado:
00c7df5188341d684206c73b31275d31c7c31cf7

# A imagem publicada pelo pipeline funcionou igual à imagem buildada localmente?
# Sim, após configurar corretamente o ambiente com docker compose.
# A aplicação subiu com três serviços: api, nginx e db/postgres.
# A API ficou disponível internamente na porta 8000, e o acesso externo foi feito
# pelo Nginx na porta 80.
# Por isso o teste correto foi:
# curl http://localhost/
# curl http://localhost/pratos/

# Essa imagem publicada pelo CI é diferente da que você buildou
# localmente no Bloco 9 (bella-tavola:v1)?
# O que garante que são equivalentes?
# Foi gerada a partir do mesmo Dockerfile,
# do mesmo código-fonte versionado e do commit identificado pelo SHA.
# O SHA permite rastrear exatamente qual versão do código originou a imagem.

# Imagine que você precisa saber qual versão do código está rodando
# em produção. Você tem acesso à tag da imagem (o SHA).
# Como você usaria o SHA para encontrar o estado exato do código?
# Eu usaria o SHA para localizar o commit correspondente no GitHub.
# Com esse commit, consigo ver exatamente os arquivos daquele momento.
# Também poderia reproduzir localmente com:
# git checkout 00c7df5188341d684206c73b31275d31c7c31cf7

<details>
<summary>💡 Gabarito</summary>

```python
# A imagem publicada pelo CI e a buildada localmente são equivalentes
# porque partem do mesmo Dockerfile e do mesmo código no mesmo commit.
# O que garante a equivalência:
# 1. O CI usa actions/checkout@v4 — baixa exatamente o commit que foi feito push
# 2. O Dockerfile é o mesmo arquivo no repositório
# 3. O requirements.txt é o mesmo arquivo no repositório
# A única diferença possível: a máquina do CI é mais limpa que a sua
# (sem cache acumulado de builds anteriores). Isso geralmente produz
# uma imagem levemente diferente em bytes (timestamps, ordem de arquivos)
# mas funcionalmente idêntica.

# Para encontrar o estado do código a partir do SHA:
# git log --oneline | grep <SHA>  → mostra a mensagem do commit
# git show <SHA>                  → mostra as alterações daquele commit
# git checkout <SHA>              → volta para aquele estado exato
# No GitHub: https://github.com/usuario/bella-tavola/commit/<SHA>
#
# Essa é a rastreabilidade completa:
# imagem em produção → SHA → commit → código exato → autor → data
# É por isso que usar o SHA como tag é a prática padrão em produção.
```

</details>

## Exercício 13.5 — Desafio: pipeline à prova de erro 🎯

**Nível:** Desafio  
**Conceito:** Verificar que o pipeline realmente protege o registry — uma imagem quebrada nunca deve ser publicada.

### Sua tarefa

O objetivo deste exercício é confirmar que o pipeline funciona como guardião: se o código está quebrado, a imagem **não deve chegar ao Docker Hub**.

**Passo 1:** Introduza um erro deliberado na API — algo que faça os testes falhar. Exemplos:
- Remova uma validação de um modelo Pydantic (um teste de 422 vai falhar)
- Altere o status code de uma rota (um teste de contrato vai falhar)
- Quebre um import em `main.py` (todos os testes vão falhar com ERROR)

**Passo 2:** Faça commit e push para o `main`.

In [ ]:
# ✏️ Observe o pipeline na aba Actions e anote:

# Qual job falhou?
# O job de integração/testes falhou após a introdução proposital de um erro no main.py.

# O job 'docker' foi executado?
# Não. O job docker não foi executado porque depende da etapa de integração/testes.

# Uma nova imagem foi publicada no Docker Hub?
# Não. Como o job docker foi bloqueado, nenhuma imagem quebrada foi publicada.

# A tag :latest foi atualizada para o commit com o erro?
# Não. A tag latest permaneceu apontando para a última imagem válida.

# Para refletir: em um time real, o que poderia dar errado se
# o job 'docker' rodasse em PARALELO com 'integracao'
# sem o 'needs: integracao'?
# Uma imagem quebrada poderia ser publicada antes de os testes terminarem.
# Mesmo que a integração falhasse depois, a imagem já estaria disponível no Docker Hub,
# podendo ser usada em produção ou em outro ambiente.

<details>
<summary>💡 Gabarito</summary>

```python
# Com o erro introduzido:
# - O job 'qualidade' ou 'integracao' falhou (dependendo do tipo de erro)
# - O job 'docker' NÃO foi executado — está cancelado/skipped
# - O Docker Hub NÃO recebeu uma nova imagem quebrada
# - A tag :latest continua apontando para a última imagem válida
#
# Isso é exatamente o propósito do 'needs: integracao':
# o pipeline é um portão — código que não passa nos testes
# nunca chega ao registry de produção.

# Se docker rodasse em paralelo com integracao:
# Cenário de corrida (race condition):
# 1. integracao começa e encontra um bug
# 2. docker começa ao mesmo tempo, faz build e push
# 3. a imagem quebrada chega ao Docker Hub
# 4. integracao falha — mas já é tarde
# 5. a tag :latest agora aponta para código quebrado
# 6. qualquer deploy automático usaria essa imagem quebrada
#
# A sequência qualidade → integracao → docker não é burocracia.
# É a garantia de que qualidade > velocidade de publicação.
```

</details>

## Erros comuns neste bloco

| Mensagem no log | Causa provável | Solução |
|---|---|---|
| `denied: requested access to the resource is denied` | Não autenticado, nome de repositório errado, ou token sem permissão de escrita | Confirme que `DOCKER_USERNAME` e `DOCKER_PASSWORD` estão corretos e que o token tem permissão **Read & Write** |
| `name unknown: Repository not found` | O repositório `bella-tavola` não existe no Docker Hub | Crie o repositório manualmente em hub.docker.com antes do push |
| `unauthorized: authentication required` | Secret `DOCKER_PASSWORD` expirado ou token revogado | Gere um novo token de acesso no Docker Hub e atualize o secret no GitHub |
| `Error: buildx: exit status 1` | Erro no Dockerfile durante o build no CI | Leia o log completo do step — geralmente um COPY falhando ou dependência faltando |
| Job `docker` aparece como `skipped` | O job anterior (`integracao`) falhou ou a condição `if` não foi satisfeita | Verifique se está fazendo push para `main` (não para outro branch) e se `integracao` passou |
| `Error: Cannot perform an interactive login from a non TTY device` | Usando `docker login` com `run:` sem `--password-stdin` | Use `docker/login-action@v3` em vez de `docker login` manual |

## Checklist do Bloco 13

- [ ] Conta no Docker Hub criada e repositório `bella-tavola` existe
- [ ] Token de acesso gerado (não a senha) e salvo como `DOCKER_PASSWORD`
- [ ] Secrets `DOCKER_USERNAME` e `DOCKER_PASSWORD` configurados no GitHub
- [ ] Job `docker` adicionado ao `ci.yml` com `needs: integracao`
- [ ] Job `relatorio` atualizado para `needs: docker`
- [ ] Pipeline verde com quatro jobs: `qualidade → integracao → docker → relatorio`
- [ ] Imagem publicada no Docker Hub com tag do SHA do commit
- [ ] Consegui fazer pull da imagem publicada e rodar localmente
- [ ] Confirmei que código quebrado não chega ao Docker Hub

---

# Mapa do que foi construído

| Arquivo | Modificado neste caderno | O que faz |
|---|---|---|
| `.github/workflows/ci.yml` | Bloco 13 | Pipeline completo com job `docker` — build e push para Docker Hub a cada merge no `main` |

---

# Checklist de competências — e05-p03

**Bloco 13 — CI com Docker**
- [ ] Sei a diferença entre senha e token de acesso e por que usar tokens em pipelines
- [ ] Configuro secrets no GitHub e os uso corretamente com `${{ secrets.NOME }}`
- [ ] O job `docker` faz build e push após os testes de integração passarem
- [ ] A imagem publicada tem o hash do commit como tag
- [ ] Consigo fazer pull da imagem publicada e rodar localmente
- [ ] Entendo por que o job `docker` depende de `integracao` (e não roda em paralelo)
- [ ] Sei o que acontece com o Docker Hub quando os testes falham

---

# Visão completa do que foi construído na série e05

Ao longo dos três cadernos, a API Bella Tavola passou de "funciona na minha máquina" para um artefato publicado, versionado e integrado ao pipeline de CI:

| Caderno | O que foi construído |
|---|---|
| **e05-p01** | `Dockerfile` — a API roda em um contêiner local reproduzível |
| **e05-p02** | `.dockerignore`, `docker-compose.yml`, `nginx.conf`, `Dockerfile` refatorado — sistema multi-serviço seguro e eficiente |
| **e05-p03** | `ci.yml` atualizado — imagem publicada automaticamente no Docker Hub a cada merge |

O pipeline completo:

```
Push para main
      ↓
  qualidade    → formatação + testes smoke
      ↓
  integracao   → modelo do Hub + testes de integração
      ↓
  docker       → build + push para Docker Hub com tag do commit
      ↓
  relatorio    → resumo com URL da imagem publicada
```